# Plotting the skeletons of every neuron in a module 

This notebook loads the module data for the subconnectome composed of oviIN_R's inputs, fetches the synaptic weights from these neurons to the oviIN_R, and plots the skeletons of the strongest oviIN_R inputs by module. 

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import bokeh
from bokeh.plotting import figure, output_notebook, show, output_file, gridplot
from bokeh.io import export_svg, export_png
from neuprint import Client, fetch_adjacencies

In [ ]:
from neuprint import Client

auth_token_file = open("flybrain.auth.txt", 'r') # need file containing the authentication token
auth_token = next(auth_token_file).strip()
try:
    np_client = Client('neuprint.janelia.org', dataset='hemibrain:' + 'v1.2.1', token=auth_token)
    np_client.fetch_version()
except:
    np_client = None

In [ ]:
# load the module data
ovi_node_df  = pd.read_csv('modularity_runs/0.0/0-0_98765.txt', header=None, sep=' ', names=['id', "0.0"])

## For thresholding
Obtain connections to oviIN_R. Right off the bat, we threshold connection weights less than 3 since those may be erroneous according to Scheffer et al. 

In [ ]:
# oviINr bodyId
oviINr = 423101189 

In [ ]:
# fetch connections to oviINr
from neuprint import fetch_simple_connections

ovi_conns = fetch_simple_connections(None, oviINr, min_weight=3)
ovi_conns

In [ ]:
# quick look at distribution of weights

plt.figure(figsize=(10, 5))
plt.title('Distribution of Connection Weights to oviINr')
plt.xlabel('Weight')
plt.ylabel('neuron count')
plt.hist(ovi_conns['weight'], bins=100)

In [ ]:
# merge the module data onto the connections data. some mod values will be nan.
ovi_mod_conns = ovi_conns[['bodyId_pre', 'type_pre', 'weight']].merge(ovi_node_df, left_on='bodyId_pre', right_on='id', how='left')

# drop the 'id' column as it's no longer needed
ovi_mod_conns = ovi_mod_conns.drop(columns=['id'])
# rename column
ovi_mod_conns = ovi_mod_conns.rename(columns={'bodyId_pre': 'id'})

ovi_mod_conns

In [ ]:
# threshold connections at 10
strong_ovi_mod_conns = ovi_mod_conns[ovi_mod_conns['weight'] > 10]
strong_ovi_mod_conns

In [ ]:
strong_ovi_mod_conns['0.0'].value_counts()

In [ ]:
# get only top 15 neurons with most connections for each cluster
top_15 = strong_ovi_mod_conns.groupby('0.0').apply(lambda x: x.nlargest(15, 'weight')).reset_index(drop=True)
top_15

In [ ]:
# quick sanity check
strong_ovi_mod_conns[strong_ovi_mod_conns['0.0']==7].head(15)

Note that while the FS1A are the strongest inputs to oviINr as a type, none are in the top 15 in their module. For that reason, I think it makes sense to be more inclusive in the skeleton plots rather than only plotting the top 15 neurons in each module.

In [ ]:
top_15[top_15['0.0']==4].head(15)

In [ ]:
strong_ovi_mod_conns[strong_ovi_mod_conns['type_pre']=='FS1A'].head(15)

## skeleton plots by module

In [ ]:
import bokeh
import bokeh.palettes
from bokeh.plotting import figure
from bokeh.layouts import gridplot
from bokeh.io import show, output_notebook, export_svg
from bokeh.resources import INLINE
output_notebook(INLINE)

In [ ]:
# create a function to choose which module to return all skeletons for
def get_and_plot(synapse_plot, cluster_id):
    cluster_curr = synapse_plot[synapse_plot['0.0']== cluster_id]
    
    # get all skeletons 
    all_skeletons = []
    # cretae skeleton for both oviINs
    for id in cluster_curr['id'].unique():
        skeletons = []
        s = np_client.fetch_skeleton(id, format='pandas')
        s['bodyId'] = id
        s['color'] = synapse_plot[synapse_plot['id']==id]['color'].values[0] # Use module color
        skeletons.append(s)

        skeletons = pd.concat(skeletons, ignore_index=True)
        # Join parent nodes
        segments = skeletons.merge(skeletons, left_on=['bodyId', 'link'], right_on=['bodyId', 'rowId'], suffixes=['_child', '_parent'])
        all_skeletons.append(segments)

    return all_skeletons

In [ ]:
# map on colors
color_dict = {1: '#4e90d3', 2: '#9467bd', 3: '#e7cf57', 4: '#ff6a88', 5: '#5cc9ff', 6: '#3a9f82', 7: '#9fad2b'}

# for all data use ovi_clusters
ovi_mod_conns['color'] = ovi_mod_conns['0.0'].map(color_dict)

# for strong connections use strong_ovi_mod_conns
strong_ovi_mod_conns['color'] = strong_ovi_mod_conns['0.0'].map(color_dict)

# top15
top_15['color'] = top_15['0.0'].map(color_dict)

In [ ]:
# get skeletons by cluster
skel = get_and_plot(strong_ovi_mod_conns, 4)

In [ ]:
# skeleton number sanity check
#len(top_ovi_conn[top_ovi_conn['0.0']==4]['id'].unique()) == len(skel)
len(skel)

In [ ]:
from bokeh.models import Range1d
from bokeh.plotting import figure, show

pmpre = figure(width=850, height=800) #, title="Synaptic input sites on oviINr colored by coarse oviINr input module")
pmpre.y_range.flipped = True

pmpre.output_backend = "svg"

for segments in skel[:]:  # Limit to first two skeletons for demonstration
    pmpre.segment(x0='x_child', x1='x_parent',
                y0='z_child', y1='z_parent',
                color='color_child',
                source=segments)

# default point size is 4

pmpre.xaxis.visible = False
#pmpre.xgrid.visible = False

pmpre.yaxis.visible = False
#pmpre.ygrid.visible = False
pmpre.x_range = Range1d(-5000, 40000)
pmpre.y_range = Range1d(40000,0)

In [ ]:
show(pmpre)

In [ ]:
from selenium import webdriver
from bokeh.io.export import export_svg

driver = webdriver.Chrome()  # or webdriver.Firefox()
export_svg(pmpre, filename="figures/mod_skeletons/mod4_skeletons.svg", webdriver=driver)
driver.quit()

In [ ]:
# save p as svg
pmpre.output_backend = "svg"

export_svg(pmpre, filename="figures/mod_skeletons/mod4_skeletons.svg")

In [ ]:
# also export png
from bokeh.io import export_png
export_png(pmpre, filename="figures/mod_skeletons/mod4_skeletons.png")